In [ ]:
# CP-style self-supervised training from a 5-channel NPY (inline Python, no shell commands)
from pathlib import Path
import os
import sys
import importlib
from types import SimpleNamespace
import numpy as np
import torch

In [ ]:
# Paths and run settings
REPO_DIR = Path("/Users/svo/source/dinov2")
NPY_PATH = Path("/scratch/svo/morpho_analysis/toy_dataset/MIPFTMA14/aveolar/processed_img_with_mask.npy")
CONFIG_FILE = REPO_DIR / "dinov2/configs/ssl_default_config.yaml"
OUTPUT_DIR = REPO_DIR / "outputs/cp_npy5_run1"

# Dataset/training settings
NUM_CHANNELS = 5
EPOCHS = 15
BATCH_SIZE_PER_GPU = 8
NUM_WORKERS = 32

# CP protocol knobs from the paper.
GLOBAL_CROP_SIZE = 128
LOCAL_CROPS_NUMBER = 0  # discard local crops for Cell Painting
EVAL_CENTER_CROP_SIZE = 128

assert REPO_DIR.exists(), f"Missing repo dir: {REPO_DIR}"
assert NPY_PATH.exists(), f"Missing dataset: {NPY_PATH}"
assert CONFIG_FILE.exists(), f"Missing config: {CONFIG_FILE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

arr = np.load(NPY_PATH, mmap_mode="r")
print("Dataset shape:", arr.shape, "dtype:", arr.dtype, "min:", arr.min(), "max:", arr.max())
if arr.ndim != 4 or arr.shape[1] < NUM_CHANNELS:
    raise ValueError(f"Expected [N,C,H,W] with C>={NUM_CHANNELS}, got {arr.shape}")
if arr.shape[-1] < GLOBAL_CROP_SIZE or arr.shape[-2] < GLOBAL_CROP_SIZE:
    raise ValueError(
        f"Input spatial size must be >= {GLOBAL_CROP_SIZE} for CP protocol, got {arr.shape[-2:]}"
    )

OFFICIAL_EPOCH_LENGTH = int(np.ceil(arr.shape[0] / BATCH_SIZE_PER_GPU))
WARMUP_EPOCHS = max(0, min(8, EPOCHS - 1))
WARMUP_TEACHER_TEMP_EPOCHS = max(0, min(8, EPOCHS - 1))

print(
    f"Run settings -> N={arr.shape[0]}, channels={NUM_CHANNELS}, epochs={EPOCHS}, "
    f"epoch_len={OFFICIAL_EPOCH_LENGTH}, warmup={WARMUP_EPOCHS}, "
    f"global_crop={GLOBAL_CROP_SIZE}, local_crops={LOCAL_CROPS_NUMBER}, "
    f"eval_center_crop={EVAL_CENTER_CROP_SIZE}"
)

In [ ]:
# Runtime dataset + training patches (keeps repo files untouched)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import random
import dinov2.train.train as train_mod
train_mod = importlib.reload(train_mod)
import dinov2.distributed as distributed
from torch.utils.data import Dataset
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP, StateDictType
from dinov2.fsdp import FSDPCheckpointer as OrigFSDPCheckpointer, rankstr
import dinov2.layers.block as block_mod

for key in [
    "SLURM_JOB_ID",
    "SLURM_JOB_NUM_NODES",
    "SLURM_JOB_NODELIST",
    "SLURM_PROCID",
    "SLURM_NTASKS",
    "SLURM_LOCALID",
]:
    os.environ.pop(key, None)

# xFormers fast index_select_cat can fail for some shapes/devices; use a safe fallback.
if not hasattr(block_mod, "_orig_index_select_cat"):
    block_mod._orig_index_select_cat = block_mod.index_select_cat

def _safe_index_select_cat(sources, indices):
    return torch.cat([s[i.long()].flatten() for s, i in zip(sources, indices)], dim=0)

block_mod.index_select_cat = _safe_index_select_cat

class NPYCPDataset(Dataset):
    def __init__(self, root, num_channels=5, transform=None, target_transform=None):
        self.arr = np.load(root, mmap_mode="r")
        self.num_channels = int(num_channels)
        self.transform = transform
        self.target_transform = target_transform

        if self.arr.ndim != 4:
            raise ValueError(f"Expected [N,C,H,W], got {self.arr.shape}")
        if self.arr.shape[1] < self.num_channels:
            raise ValueError(f"Expected at least {self.num_channels} channels, got {self.arr.shape[1]}")

    def __len__(self):
        return int(self.arr.shape[0])

    def __getitem__(self, idx):
        image = np.asarray(self.arr[idx, : self.num_channels]).astype(np.float32, copy=False)
        img_max = float(np.max(image)) if image.size > 0 else 0.0
        if img_max > 255.0:
            image = image * (255.0 / 65535.0)
        image = torch.from_numpy(image)
        target = 0

        if self.transform is not None:
            image = self.transform(image)
        if self.target_transform is not None:
            target = self.target_transform(target)
        return image, target

def _parse_dataset_str_runtime(dataset_str: str):
    tokens = dataset_str.split(":")
    name = tokens[0]
    kwargs = {}
    for token in tokens[1:]:
        k, v = token.split("=", 1)
        kwargs[k] = v
    return name, kwargs

if not hasattr(train_mod, "_orig_make_dataset_for_npycp"):
    train_mod._orig_make_dataset_for_npycp = train_mod.make_dataset

def _make_dataset_runtime(*, dataset_str, transform=None, target_transform=None):
    name, kwargs = _parse_dataset_str_runtime(dataset_str)
    if name == "NPYCP":
        root = kwargs["root"]
        num_channels = int(kwargs.get("wildcard", "5"))
        return NPYCPDataset(
            root=root,
            num_channels=num_channels,
            transform=transform,
            target_transform=target_transform,
        )
    return train_mod._orig_make_dataset_for_npycp(
        dataset_str=dataset_str,
        transform=transform,
        target_transform=target_transform,
    )

train_mod.make_dataset = _make_dataset_runtime

# Custom collate that supports CP protocol with local_crops_number = 0.
def _collate_data_and_cast_fp32(
    samples_list, mask_ratio_tuple, mask_probability, dtype, n_tokens=None, mask_generator=None
):
    n_global_crops = len(samples_list[0][0]["global_crops"])
    n_local_crops = len(samples_list[0][0]["local_crops"])

    collated_global_crops = torch.stack(
        [s[0]["global_crops"][i] for i in range(n_global_crops) for s in samples_list]
    )

    if n_local_crops > 0:
        collated_local_crops = torch.stack(
            [s[0]["local_crops"][i] for i in range(n_local_crops) for s in samples_list]
        )
    else:
        # xFormers nested path cannot handle an actually empty branch; provide a surrogate local crop.
        collated_local_crops = torch.stack([s[0]["global_crops"][0] for s in samples_list])

    B = len(collated_global_crops)
    N = n_tokens
    n_samples_masked = int(B * mask_probability)
    probs = torch.linspace(*mask_ratio_tuple, n_samples_masked + 1)
    upperbound = 0
    masks_list = []
    for i in range(0, n_samples_masked):
        prob_min = probs[i]
        prob_max = probs[i + 1]
        masks_list.append(torch.BoolTensor(mask_generator(int(N * random.uniform(prob_min, prob_max)))))
        upperbound += int(N * prob_max)
    for _ in range(n_samples_masked, B):
        masks_list.append(torch.BoolTensor(mask_generator(0)))

    random.shuffle(masks_list)

    collated_masks = torch.stack(masks_list).flatten(1)
    mask_indices_list = collated_masks.flatten().nonzero().flatten()
    masks_weight = (1 / collated_masks.sum(-1).clamp(min=1.0)).unsqueeze(-1).expand_as(collated_masks)[
        collated_masks
    ]

    return {
        "collated_global_crops": collated_global_crops.to(torch.float32),
        "collated_local_crops": collated_local_crops.to(torch.float32),
        "collated_masks": collated_masks,
        "mask_indices_list": mask_indices_list,
        "masks_weight": masks_weight,
        "upperbound": upperbound,
        "n_masked_patches": torch.full((1,), fill_value=mask_indices_list.shape[0], dtype=torch.long),
    }

train_mod.collate_data_and_cast = _collate_data_and_cast_fp32

class JupyterFSDPCheckpointer(OrigFSDPCheckpointer):
    def save(self, name: str, **kwargs):
        if not self.save_dir or not self.save_to_disk:
            return
        data = {}
        with FSDP.state_dict_type(self.model, StateDictType.FULL_STATE_DICT):
            data["model"] = self.model.state_dict()
        for key, obj in self.checkpointables.items():
            data[key] = obj.state_dict()
        data.update(kwargs)
        basename = f"{name}.{rankstr()}.pth"
        save_file = os.path.join(self.save_dir, basename)
        with self.path_manager.open(save_file, "wb") as f:
            torch.save(data, f)
        self.tag_last_checkpoint(basename)

    def load(self, *args, **kwargs):
        with FSDP.state_dict_type(self.model, StateDictType.FULL_STATE_DICT):
            return super().load(*args, **kwargs)

train_mod.FSDPCheckpointer = JupyterFSDPCheckpointer
print("Runtime patching complete: NPYCP dataset + xFormers-safe collate + notebook-safe checkpointer")

In [ ]:
# Build training opts for CP-style (5-channel) self-supervised run
dataset_opt = f"train.dataset_path=NPYCP:root={NPY_PATH}:wildcard={NUM_CHANNELS}"
opts = [
    dataset_opt,
    f"train.num_workers={NUM_WORKERS}",
    f"train.batch_size_per_gpu={BATCH_SIZE_PER_GPU}",
    f"train.OFFICIAL_EPOCH_LENGTH={OFFICIAL_EPOCH_LENGTH}",
    "train.cell_augmentation=true",
    f"optim.epochs={EPOCHS}",
    f"optim.warmup_epochs={WARMUP_EPOCHS}",
    f"teacher.warmup_teacher_temp_epochs={WARMUP_TEACHER_TEMP_EPOCHS}",
    "evaluation.eval_period_iterations=0",
    "student.arch=vit_small",
    "student.patch_size=8",
    f"student.in_chans={NUM_CHANNELS}",
    f"teacher.in_chans={NUM_CHANNELS}",
    f"crops.global_crops_size={GLOBAL_CROP_SIZE}",
    f"crops.local_crops_number={LOCAL_CROPS_NUMBER}",
    # Avoid xFormers index_select_cat block-size constraint path on this setup.
    "student.drop_path_rate=0.0",
    # Full precision for 5-channel stability in notebook runs.
    "compute_precision.grad_scaler=false",
    "compute_precision.student.backbone.mixed_precision.param_dtype=fp32",
    "compute_precision.student.backbone.mixed_precision.reduce_dtype=fp32",
    "compute_precision.student.backbone.mixed_precision.buffer_dtype=fp32",
    "compute_precision.teacher.backbone.mixed_precision.param_dtype=fp32",
    "compute_precision.teacher.backbone.mixed_precision.reduce_dtype=fp32",
    "compute_precision.teacher.backbone.mixed_precision.buffer_dtype=fp32",
]

for item in opts:
    print(item)

In [ ]:
# Start training (inline, no shell command)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this training run.")
if distributed.is_enabled():
    raise RuntimeError(
        "Distributed process group is already initialized in this kernel. "
        "Restart kernel before running this cell again."
    )

args = SimpleNamespace(
    config_file=str(CONFIG_FILE),
    no_resume=True,
    eval_only=False,
    eval="",
    opts=list(opts),
    output_dir=str(OUTPUT_DIR),
    seed=0,
)

cfg = train_mod.setup(args)
model = train_mod.SSLMetaArch(cfg).to(torch.device("cuda"))
model.prepare_for_distributed_training()
train_mod.do_train(cfg, model, resume=not args.no_resume)

print(f"Training completed. Outputs saved to: {OUTPUT_DIR}")

In [ ]:
# Optional: inspect produced checkpoints
eval_ckpts = sorted((OUTPUT_DIR / "eval").glob("training_*/teacher_checkpoint.pth")) if (OUTPUT_DIR / "eval").exists() else []
model_ckpts = sorted(OUTPUT_DIR.glob("model_*.rank_0.pth"))

print("Teacher eval checkpoints:", len(eval_ckpts))
if eval_ckpts:
    print("Latest teacher checkpoint:", eval_ckpts[-1])
print("Model checkpoints:", len(model_ckpts))
if model_ckpts:
    print("Latest model checkpoint:", model_ckpts[-1])

In [ ]:
# Load trained backbone for embedding extraction (teacher branch)
from omegaconf import OmegaConf
from dinov2.configs import dinov2_default_config
from dinov2.models import build_model_from_cfg
from dinov2.data.cell_dino.transforms import make_classification_eval_cell_transform, NormalizationType

def _build_eval_backbone():
    embed_opts = [
        "student.arch=vit_small",
        "student.patch_size=8",
        f"student.in_chans={NUM_CHANNELS}",
        f"teacher.in_chans={NUM_CHANNELS}",
        f"crops.global_crops_size={GLOBAL_CROP_SIZE}",
        "student.drop_path_rate=0.0",
    ]
    cfg_embed = OmegaConf.merge(
        OmegaConf.create(dinov2_default_config),
        OmegaConf.load(CONFIG_FILE),
        OmegaConf.from_dotlist(embed_opts),
    )
    backbone, _ = build_model_from_cfg(cfg_embed, only_teacher=True)
    return backbone

def _load_teacher_backbone_weights(backbone, output_dir: Path):
    eval_ckpts = sorted((output_dir / "eval").glob("training_*/teacher_checkpoint.pth"))
    if eval_ckpts:
        ckpt_path = eval_ckpts[-1]
        state = torch.load(ckpt_path, map_location="cpu")
        teacher_sd = state.get("teacher", state)
        teacher_sd = {k.replace("module.", "").replace("backbone.", "", 1): v for k, v in teacher_sd.items()}
        msg = backbone.load_state_dict(teacher_sd, strict=False)
        return ckpt_path, msg

    model_ckpts = sorted(output_dir.glob("model_*.rank_0.pth"))
    if not model_ckpts:
        raise FileNotFoundError(
            f"No checkpoint found in {output_dir}. Run training first, then rerun this cell."
        )

    ckpt_path = model_ckpts[-1]
    state = torch.load(ckpt_path, map_location="cpu")
    model_sd = state.get("model", state)

    teacher_prefixes = (
        "teacher.backbone.",
        "module.teacher.backbone.",
        "teacher.backbone.module.",
    )
    teacher_sd = {}
    for k, v in model_sd.items():
        for p in teacher_prefixes:
            if k.startswith(p):
                teacher_sd[k[len(p) :]] = v
                break

    if not teacher_sd:
        raise RuntimeError(
            f"Could not find teacher backbone keys in checkpoint: {ckpt_path}"
        )

    msg = backbone.load_state_dict(teacher_sd, strict=False)
    return ckpt_path, msg

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedding_model = _build_eval_backbone()
embedding_ckpt_path, embedding_load_msg = _load_teacher_backbone_weights(embedding_model, OUTPUT_DIR)
embedding_model = embedding_model.to(device).eval()

embedding_transform = make_classification_eval_cell_transform(
    crop_size=EVAL_CENTER_CROP_SIZE,
    normalization_type=NormalizationType.SELF_NORM_CENTER_CROP,
)

print("Embedding model ready")
print("Checkpoint:", embedding_ckpt_path)
print("Load status:", embedding_load_msg)

In [ ]:
# Batch embedding extraction for full NPY array: [N, C, H, W] -> [N, hidden_dim]
from torch.utils.data import DataLoader, Dataset

EMBED_NPY_PATH = NPY_PATH  # set another file path here if needed
EMBED_BATCH_SIZE = 64
EMBED_NUM_WORKERS = min(8, os.cpu_count() or 1)
EMBED_CROP_SIZE = 128  # keep 128 for training-consistent embeddings; set 160 to use full input
OUTPUT_EMBED_PATH = OUTPUT_DIR / "embeddings_cls.npy"

arr_embed = np.load(EMBED_NPY_PATH, mmap_mode="r")
if arr_embed.ndim != 4:
    raise ValueError(f"Expected [N,C,H,W], got {arr_embed.shape}")
if arr_embed.shape[1] < NUM_CHANNELS:
    raise ValueError(f"Need at least {NUM_CHANNELS} channels, got {arr_embed.shape[1]}")
if arr_embed.shape[-2] < EMBED_CROP_SIZE or arr_embed.shape[-1] < EMBED_CROP_SIZE:
    raise ValueError(
        f"Input spatial size must be >= {EMBED_CROP_SIZE}, got {arr_embed.shape[-2:]}"
    )

batch_embedding_transform = make_classification_eval_cell_transform(
    crop_size=EMBED_CROP_SIZE,
    normalization_type=NormalizationType.SELF_NORM_CENTER_CROP,
)

class NPYEvalDataset(Dataset):
    def __init__(self, arr, num_channels, transform):
        self.arr = arr
        self.num_channels = int(num_channels)
        self.transform = transform

    def __len__(self):
        return int(self.arr.shape[0])

    def __getitem__(self, idx):
        x = np.asarray(self.arr[idx, : self.num_channels]).astype(np.float32, copy=False)
        x_max = float(np.max(x)) if x.size > 0 else 0.0
        if x_max > 255.0:
            x = x * (255.0 / 65535.0)
        x = torch.from_numpy(x)
        x = self.transform(x)
        return x

eval_ds = NPYEvalDataset(arr_embed, NUM_CHANNELS, batch_embedding_transform)
eval_loader = DataLoader(
    eval_ds,
    batch_size=EMBED_BATCH_SIZE,
    shuffle=False,
    num_workers=EMBED_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

num_samples = len(eval_ds)
hidden_dim = int(getattr(embedding_model, "embed_dim", 0))
if hidden_dim <= 0:
    raise RuntimeError("Could not infer hidden dimension from embedding_model.embed_dim")

embeddings_cls = np.empty((num_samples, hidden_dim), dtype=np.float32)

embedding_model.eval()
offset = 0
with torch.no_grad():
    for step, batch in enumerate(eval_loader, start=1):
        batch = batch.to(device, non_blocking=True)
        cls_batch = embedding_model.forward_features(batch)["x_norm_clstoken"]
        cls_batch = cls_batch.detach().cpu().numpy().astype(np.float32, copy=False)

        bsz = cls_batch.shape[0]
        embeddings_cls[offset : offset + bsz] = cls_batch
        offset += bsz

        if step == 1 or step % 10 == 0 or step == len(eval_loader):
            print(f"Processed {offset}/{num_samples} images")

if offset != num_samples:
    raise RuntimeError(f"Embedding count mismatch: expected {num_samples}, got {offset}")

np.save(OUTPUT_EMBED_PATH, embeddings_cls)
print("Saved CLS embeddings:", OUTPUT_EMBED_PATH)
print("Embeddings shape:", embeddings_cls.shape, "(N, hidden_dim)")